In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt

from mach3sbitools.simulator import Simulator
from mach3sbitools.utils import from_feather

# The Command Line Interface
The main interface you should be using with MaCh3 SBI Tools is in the command line! The main command is
```bash
mach3sbi
```

You can list sub-commands and get help for those with `--help`

## `mach3sbi simulate`
This will generate you N simulations and save them to a feather file! It can also save your prior to external file which is needed during the ML training!
```sh
mach3sbi simulate -m "my_simulator" -s "MySimulator" -c physics_configs/PhysicsConfig.yaml -o data/my_data.feather -n 300000 -r "prior/my_prior.pkl"
```


In [ ]:
# Let's first load back in your simulator
# First things first, let's use the MaCh3SBITools Simulator instance

# Our config file
physics_config = Path("physics_configs/PhysicsConfig.yaml")

# We also let it know that our cyclical parameter is cyclical!
mach3sbi_simulator = Simulator("my_simulator", "MySimulator", physics_config)

In [ ]:
# Once you've run this in terminal let's check they exist!
output = Path("data/my_data.feather")

if not output.is_file():
    raise FileExistsError("Cannot find the file, make sure the output path is correct!")

# We can read a single simulation file straight back in
# Remember: theta == your systematics, x==your simulations!
theta, x = from_feather(output)

In [ ]:
# Now let's do a quick pair plot of our sims to check they're okay!
from sbi.analysis import pairplot

# Firstly the parameter plot (the names need to be inverted a bit)
pairplot(
    theta, labels=[[n] for n in mach3sbi_simulator.prior.prior_data.parameter_names]
)

plt.show()

## `mach3sbi save_data`
This simply saves your data histogram so you don't need to open it later! Let's save it to `nominal_data.parquet`
```sh
mkdir observations
mach3sbi save_data -m "my_simulator" -s "MySimulator" -c physics_configs/PhysicsConfig.yaml --cyclical_pars cyclical_param  -o observations/observed_data.paraquet
```


In [ ]:
# We can look at the save data with Paraquet
import numpy as np
from pyarrow import parquet

table_file = Path("observations/observed_data.paraquet")
if not table_file.is_file():
    raise FileExistsError(
        f"Please run mach3sbi save_data -m 'my_simulator' -s 'MySimulator' -c physics_configs/PhysicsConfig.yaml --cyclical_pars cyclical_param  -o {table_file!s}"
    )

table = parquet.read_table(table_file)
table_arr = np.array(table["data"])


# We can plot the histogram of the data as well (even if we don't know the bin edges!)
plt.stairs(table_arr)
plt.show()

## `mach3sbi merge-shards`
Training doesn't read the `.feather` files directly. Instead, all the simulation files in a folder are merged into two
flat arrays, `theta.npy` and `x.npy`, which can be streamed from disk efficiently however big the dataset gets.
```sh
mach3sbi merge-shards -s data/ -o training_data/ --shuffle
```

`--shuffle` mixes the rows from all the files together. This matters because the validation set is taken from the
end of the merged data, so without it validation would only ever see the last few simulation files!

If you used `--nuisance_pars` when simulating, pass the prior with `-r prior/my_prior.pkl` so the nuisance
parameters are dropped from the merged files and never have to be read during training.

In [ ]:
# Let's check the merge worked
merged = Path("training_data")
for f in ("theta.npy", "x.npy"):
    if not (merged / f).is_file():
        raise FileNotFoundError(
            f"Cannot find {merged / f}, please run mach3sbi merge-shards -s data/ -o training_data/ --shuffle"
        )

# The merged files can be memory-mapped, so this doesn't load them into RAM
theta_merged = np.load(merged / "theta.npy", mmap_mode="r")
x_merged = np.load(merged / "x.npy", mmap_mode="r")
print(f"theta: {theta_merged.shape} | x: {x_merged.shape}")

With our observations saved and your simulations merged, we're now ready to train an SBI algorithm!